# Notebook 04b — SHAP Raw vs Calibrated Validation Experiment

**Purpose:** Validate that feature attribution rankings (SHAP-based) are robust to hybrid calibration. Tests whether the design choice in Notebook 04 — computing SHAP on raw model outputs rather than calibrated outputs — distorts the interpretability claims.

**Method:** For 9 selected models (3 datasets × 3 architectures, all CW variants), compute SHAP values both ways and compare rankings via Spearman correlation and top-5 overlap.

**Pipelines compared:**
- **Raw (already computed in Notebook 04):** model → raw_probs → SHAP via TreeExplainer/GradientExplainer
- **Calibrated (this notebook):** model → raw_probs → hybrid_calibrator → calibrated_probs → SHAP via KernelExplainer

**Why KernelExplainer for calibrated:** Hybrid isotonic regression has non-differentiable step functions, so neither GradientExplainer (needs gradients) nor TreeExplainer (model-specific) work through the calibration step. KernelExplainer is model-agnostic and treats the entire pipeline as black-box.

**Models (9 total, all CW variants):**
- NSL-KDD: rf_5class_cw, xgb_5class_cw, dnn_5class_cw
- UNSW-NB15: rf_5class_cw, xgb_5class_cw, dnn_5class_cw
- CIC-IDS2017: rf_5class_cw, xgb_5class_cw, dnn_5class_cw

**Why CW only:** v5 calibration findings showed Dirichlet absorbs class-weight effects most strongly on CW models. If calibration is going to change SHAP rankings anywhere, it'll be on CW models. This is the strongest test.

**Sample budget:** 100 stratified test samples per model, drawn from within the 1000 already used by Notebook 04 (so SHAP_raw can be directly subsetted for comparison).

**KernelExplainer config:** 50 background samples, nsamples='auto' (defaults to 2*M+2048).

**Time estimate:** ~10-15 min per model × 9 = ~90-120 min total.

**Output:**
- `results/tables/shap_raw_vs_calibrated.csv` (per-model, per-class metrics)
- `results/tables/shap_validation_summary.json` (overall conclusions)

**Prerequisite:** Notebook 03e must have run successfully (fitted calibrators saved as `.joblib` files).

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
import joblib
import time
import traceback
from pathlib import Path
from datetime import datetime
from collections import Counter
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import shap

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device: {DEVICE}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

N_BACKGROUND = 50
N_EVAL = 100

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
MODELS = ['rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']  # 9 total

print(f'\nValidation experiment config:')
print(f'  Models: {len(DATASETS)*len(MODELS)} = {len(DATASETS)*len(MODELS)} (3 datasets × 3 architectures, all CW)')
print(f'  Background: {N_BACKGROUND} samples')
print(f'  Evaluation: {N_EVAL} samples (subset of Notebook 04 indices)')

PyTorch device: cuda

Validation experiment config:
  Models: 9 = 9 (3 datasets × 3 architectures, all CW)
  Background: 50 samples
  Evaluation: 100 samples (subset of Notebook 04 indices)


## 2. DNN architecture + calibrator wrapper

In [3]:
class DNN(nn.Module):
    """Same architecture as Notebook 04."""
    def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def load_pytorch_dnn_softmax(path):
    """Load DNN and return a callable that takes numpy X, returns softmax probs as numpy."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)

    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        in_dim = ckpt['in_dim']
        n_classes = ckpt['n_classes']
        hidden = tuple(ckpt['hidden'])
        dropout = ckpt['dropout']
        state_dict = ckpt['state_dict']
    else:
        state_dict = ckpt
        in_dim = state_dict['net.0.weight'].shape[1]
        n_classes = state_dict['net.16.weight'].shape[0]
        hidden = (256, 128, 64, 32)
        dropout = 0.3

    model = DNN(in_dim=in_dim, n_classes=n_classes, hidden=hidden, dropout=dropout)
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()

    def predict_proba_np(X_np):
        X_t = torch.from_numpy(X_np.astype(np.float32)).to(DEVICE)
        with torch.no_grad():
            logits = model(X_t)
            probs = torch.softmax(logits, dim=-1)
        return probs.cpu().numpy()
    return predict_proba_np

def apply_calibrator(calibrator, strategy, p):
    """Identical to Notebook 03e."""
    if strategy == 'isotonic':
        return calibrator.predict(p)
    else:
        return calibrator.predict_proba(p.reshape(-1, 1))[:, 1]

def make_calibrated_pipeline(raw_predict_proba, bundle):
    """Build a callable that takes X (numpy) and returns CALIBRATED probabilities.
    Used by SHAP KernelExplainer.
    """
    calibrators = bundle['calibrators']
    strategies = bundle['strategies']
    n_classes = bundle['n_classes']

    def calibrated_predict_proba(X_np):
        if X_np.ndim == 1:
            X_np = X_np.reshape(1, -1)
        raw = raw_predict_proba(X_np)
        cal = np.zeros_like(raw)
        for c in range(n_classes):
            cal[:, c] = apply_calibrator(calibrators[c], strategies[c], raw[:, c])
        row_sums = cal.sum(axis=1, keepdims=True)
        row_sums = np.where(row_sums == 0, 1, row_sums)
        return cal / row_sums
    return calibrated_predict_proba

print('✓ DNN + calibrator wrappers loaded')

✓ DNN + calibrator wrappers loaded


## 3. Path helpers and model loaders

In [4]:
def find_tree_model_path(dataset, model_name):
    base = Path(REPO) / 'models' / dataset
    for ext in ['.pkl', '.joblib']:
        p = base / f'{model_name}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'No tree model for {model_name} in {base}')

def find_dnn_path(dataset, model_name):
    p = Path(REPO) / 'models' / dataset / f'{model_name}.pt'
    if not p.exists():
        raise FileNotFoundError(f'No DNN file at {p}')
    return p

def load_raw_predict_proba(dataset, model_name):
    """Returns a callable f(X_np) -> raw probs (n, n_classes)."""
    model_type = 'rf' if 'rf' in model_name else ('xgb' if 'xgb' in model_name else 'dnn')
    if model_type in ('rf', 'xgb'):
        model = joblib.load(find_tree_model_path(dataset, model_name))
        return lambda X: model.predict_proba(X), model_type
    else:
        return load_pytorch_dnn_softmax(find_dnn_path(dataset, model_name)), 'dnn'

def load_fitted_calibrator(dataset, model_name):
    p = Path(REPO) / 'calibrators' / dataset / f'{model_name}_hybrid_fitted.joblib'
    if not p.exists():
        raise FileNotFoundError(f'Refit calibrators missing: {p}\nRun Notebook 03e first.')
    return joblib.load(p)

print('✓ Loaders ready')

✓ Loaders ready


## 4. SHAP comparison for one model

In [5]:
def compare_shap_for_model(dataset, model_name):
    """Compute calibrated SHAP for 100 test samples and compare to raw SHAP from Notebook 04.
    Returns dict with metrics.
    """
    t0 = time.time()

    # Load test data
    X_test = np.load(f'{REPO}/data/processed/{dataset}/X_test.npy').astype(np.float32)
    X_calib = np.load(f'{REPO}/data/processed/{dataset}/X_calib.npy').astype(np.float32)
    y_calib = np.load(f'{REPO}/data/processed/{dataset}/y_calib_5class.npy')
    y_test = np.load(f'{REPO}/data/processed/{dataset}/y_test_5class.npy')

    # Load existing SHAP_raw from Notebook 04 and its eval indices
    shap_raw_full = np.load(f'{REPO}/shap_values/{dataset}/{model_name}_shap.npy')  # (1000, n_feat, 5)
    eval_idx_full = np.load(f'{REPO}/shap_values/{dataset}/{model_name}_eval_idx.npy')  # (1000,)

    # Sample N_EVAL=100 from the existing 1000 indices (stratified by y_test at those indices)
    rng = np.random.default_rng(SEED + abs(hash(model_name)) % 1000)
    y_at_eval = y_test[eval_idx_full]
    n_per_class = max(1, N_EVAL // 5)
    subset_positions = []
    for c in range(5):
        class_positions = np.where(y_at_eval == c)[0]
        take = min(n_per_class, len(class_positions))
        subset_positions.append(rng.choice(class_positions, size=take, replace=False))
    subset_positions = np.concatenate(subset_positions)
    if len(subset_positions) < N_EVAL:
        remaining = list(set(range(len(eval_idx_full))) - set(subset_positions.tolist()))
        extra = rng.choice(remaining, size=N_EVAL - len(subset_positions), replace=False)
        subset_positions = np.concatenate([subset_positions, extra])
    subset_positions = np.sort(subset_positions[:N_EVAL])

    # Subset the raw SHAP arrays
    shap_raw = shap_raw_full[subset_positions]  # (100, n_feat, 5)
    eval_idx_subset = eval_idx_full[subset_positions]  # the test indices we'll use
    X_eval = X_test[eval_idx_subset]

    # Background for KernelExplainer: stratified 50 from calib set
    bg_positions = []
    n_bg_per_class = max(1, N_BACKGROUND // 5)
    for c in range(5):
        class_idx = np.where(y_calib == c)[0]
        take = min(n_bg_per_class, len(class_idx))
        bg_positions.append(rng.choice(class_idx, size=take, replace=False))
    bg_positions = np.concatenate(bg_positions)
    if len(bg_positions) < N_BACKGROUND:
        remaining = list(set(range(len(y_calib))) - set(bg_positions.tolist()))
        extra = rng.choice(remaining, size=N_BACKGROUND - len(bg_positions), replace=False)
        bg_positions = np.concatenate([bg_positions, extra])
    X_bg = X_calib[bg_positions[:N_BACKGROUND]]

    # Build calibrated pipeline
    raw_predict, model_type = load_raw_predict_proba(dataset, model_name)
    bundle = load_fitted_calibrator(dataset, model_name)
    calibrated_pipeline = make_calibrated_pipeline(raw_predict, bundle)

    # KernelExplainer on calibrated pipeline
    explainer = shap.KernelExplainer(calibrated_pipeline, X_bg)
    # Use l1_reg='num_features(10)' to speed up; 'auto' is default. We use auto.
    shap_cal_list = explainer.shap_values(X_eval, silent=True)

    # KernelExplainer returns list of (n_samples, n_features) for multiclass
    if isinstance(shap_cal_list, list):
        shap_cal = np.stack(shap_cal_list, axis=-1)  # (100, n_feat, 5)
    else:
        shap_cal = shap_cal_list
        if shap_cal.ndim == 3 and shap_cal.shape[0] == 5:
            shap_cal = np.transpose(shap_cal, (1, 2, 0))

    # METRICS
    # Global feature importance: mean |SHAP| over samples & classes
    raw_imp_global = np.mean(np.abs(shap_raw), axis=(0, 2))
    cal_imp_global = np.mean(np.abs(shap_cal), axis=(0, 2))
    rho_global, _ = spearmanr(raw_imp_global, cal_imp_global)

    raw_top5_global = set(np.argsort(raw_imp_global)[-5:])
    cal_top5_global = set(np.argsort(cal_imp_global)[-5:])
    top5_overlap_global = len(raw_top5_global & cal_top5_global)

    # Per-class
    per_class = {}
    for c in range(5):
        raw_imp_c = np.mean(np.abs(shap_raw[:, :, c]), axis=0)
        cal_imp_c = np.mean(np.abs(shap_cal[:, :, c]), axis=0)
        rho_c, _ = spearmanr(raw_imp_c, cal_imp_c)
        raw_top5 = set(np.argsort(raw_imp_c)[-5:])
        cal_top5 = set(np.argsort(cal_imp_c)[-5:])
        per_class[c] = {
            'spearman': float(rho_c) if not np.isnan(rho_c) else None,
            'top5_overlap': len(raw_top5 & cal_top5),
        }

    elapsed = time.time() - t0

    # Save calibrated SHAP for later inspection if wanted
    out_dir = Path(REPO) / 'shap_values' / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f'{model_name}_shap_calibrated.npy', shap_cal)
    np.save(out_dir / f'{model_name}_eval_idx_calibrated.npy', eval_idx_subset)

    return {
        'dataset': dataset,
        'model': model_name,
        'model_type': model_type,
        'spearman_global': float(rho_global) if not np.isnan(rho_global) else None,
        'top5_overlap_global': top5_overlap_global,
        'per_class': per_class,
        'shap_cal_shape': list(shap_cal.shape),
        'time_seconds': round(elapsed, 1),
    }

In [7]:
import os
from pathlib import Path

# Check Drive mount
test_path = '/content/drive/MyDrive/XIDS_Research/xids-research/notebooks'
print(f'Drive mounted: {os.path.exists(test_path)}')

# Check which calibrated SHAP files got saved
print('\nCalibrated SHAP outputs saved so far:')
for ds in ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']:
    d = Path(f'/content/drive/MyDrive/XIDS_Research/xids-research/shap_values/{ds}')
    cal_files = sorted([f.name for f in d.iterdir() if 'calibrated' in f.name])
    print(f'  {ds}: {cal_files}')

Drive mounted: True

Calibrated SHAP outputs saved so far:
  nsl_kdd_v2: ['rf_5class_cw_eval_idx_calibrated.npy', 'rf_5class_cw_shap_calibrated.npy']
  unsw_nb15_v2: []
  cic_ids2017_v2: []


In [8]:
# Run this BEFORE the main loop cell, in a fresh cell

import numpy as np
from pathlib import Path

# Wrap compare_shap_for_model with resumability
_original_compare = compare_shap_for_model

def compare_shap_for_model(dataset, model_name):
    """Resumable wrapper: skips if calibrated SHAP already exists for this model."""
    out_path = Path(REPO) / 'shap_values' / dataset / f'{model_name}_shap_calibrated.npy'
    eval_idx_path = Path(REPO) / 'shap_values' / dataset / f'{model_name}_eval_idx_calibrated.npy'

    if out_path.exists() and eval_idx_path.exists():
        try:
            shap_cal = np.load(out_path)
            eval_idx = np.load(eval_idx_path)
            print(f'(reusing existing) ', end='')

            # Recompute metrics from cached SHAP arrays
            shap_raw_full = np.load(f'{REPO}/shap_values/{dataset}/{model_name}_shap.npy')
            eval_idx_full = np.load(f'{REPO}/shap_values/{dataset}/{model_name}_eval_idx.npy')
            # Find positions in eval_idx_full that match eval_idx
            positions = np.array([np.where(eval_idx_full == idx)[0][0] for idx in eval_idx])
            shap_raw = shap_raw_full[positions]

            from scipy.stats import spearmanr
            raw_imp_global = np.mean(np.abs(shap_raw), axis=(0, 2))
            cal_imp_global = np.mean(np.abs(shap_cal), axis=(0, 2))
            rho_global, _ = spearmanr(raw_imp_global, cal_imp_global)

            raw_top5_global = set(np.argsort(raw_imp_global)[-5:])
            cal_top5_global = set(np.argsort(cal_imp_global)[-5:])
            top5_overlap_global = len(raw_top5_global & cal_top5_global)

            per_class = {}
            for c in range(5):
                raw_imp_c = np.mean(np.abs(shap_raw[:, :, c]), axis=0)
                cal_imp_c = np.mean(np.abs(shap_cal[:, :, c]), axis=0)
                rho_c, _ = spearmanr(raw_imp_c, cal_imp_c)
                raw_top5 = set(np.argsort(raw_imp_c)[-5:])
                cal_top5 = set(np.argsort(cal_imp_c)[-5:])
                per_class[c] = {
                    'spearman': float(rho_c) if not np.isnan(rho_c) else None,
                    'top5_overlap': len(raw_top5 & cal_top5),
                }

            model_type = 'rf' if 'rf' in model_name else ('xgb' if 'xgb' in model_name else 'dnn')
            return {
                'dataset': dataset, 'model': model_name, 'model_type': model_type,
                'spearman_global': float(rho_global) if not np.isnan(rho_global) else None,
                'top5_overlap_global': top5_overlap_global,
                'per_class': per_class,
                'shap_cal_shape': list(shap_cal.shape),
                'time_seconds': 0.0,  # cached
            }
        except Exception as e:
            print(f'(cache load failed: {e}, recomputing) ', end='')

    return _original_compare(dataset, model_name)

print('✓ Resumability wrapper installed')

✓ Resumability wrapper installed


In [ ]:
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr

_original_compare = compare_shap_for_model

def compare_shap_for_model(dataset, model_name):
    """Resumable wrapper: skips if calibrated SHAP already exists."""
    out_path = Path(REPO) / 'shap_values' / dataset / f'{model_name}_shap_calibrated.npy'
    eval_idx_path = Path(REPO) / 'shap_values' / dataset / f'{model_name}_eval_idx_calibrated.npy'

    if out_path.exists() and eval_idx_path.exists():
        try:
            shap_cal = np.load(out_path)
            eval_idx = np.load(eval_idx_path)
            print(f'(reusing existing) ', end='')

            shap_raw_full = np.load(f'{REPO}/shap_values/{dataset}/{model_name}_shap.npy')
            eval_idx_full = np.load(f'{REPO}/shap_values/{dataset}/{model_name}_eval_idx.npy')
            positions = np.array([np.where(eval_idx_full == idx)[0][0] for idx in eval_idx])
            shap_raw = shap_raw_full[positions]

            raw_imp_global = np.mean(np.abs(shap_raw), axis=(0, 2))
            cal_imp_global = np.mean(np.abs(shap_cal), axis=(0, 2))
            rho_global, _ = spearmanr(raw_imp_global, cal_imp_global)

            raw_top5_global = set(np.argsort(raw_imp_global)[-5:])
            cal_top5_global = set(np.argsort(cal_imp_global)[-5:])
            top5_overlap_global = len(raw_top5_global & cal_top5_global)

            per_class = {}
            for c in range(5):
                raw_imp_c = np.mean(np.abs(shap_raw[:, :, c]), axis=0)
                cal_imp_c = np.mean(np.abs(shap_cal[:, :, c]), axis=0)
                rho_c, _ = spearmanr(raw_imp_c, cal_imp_c)
                raw_top5 = set(np.argsort(raw_imp_c)[-5:])
                cal_top5 = set(np.argsort(cal_imp_c)[-5:])
                per_class[c] = {
                    'spearman': float(rho_c) if not np.isnan(rho_c) else None,
                    'top5_overlap': len(raw_top5 & cal_top5),
                }

            model_type = 'rf' if 'rf' in model_name else ('xgb' if 'xgb' in model_name else 'dnn')
            return {
                'dataset': dataset, 'model': model_name, 'model_type': model_type,
                'spearman_global': float(rho_global) if not np.isnan(rho_global) else None,
                'top5_overlap_global': top5_overlap_global,
                'per_class': per_class,
                'shap_cal_shape': list(shap_cal.shape),
                'time_seconds': 0.0,
            }
        except Exception as e:
            print(f'(cache load failed: {e}, recomputing) ', end='')

    return _original_compare(dataset, model_name)

print('✓ Resumability wrapper installed')

## 5. Main loop over 9 models

In [9]:
results = []
errors = []
t_overall = time.time()

print(f'\n{"="*70}')
print(f'SHAP validation experiment — {datetime.now().strftime("%H:%M:%S")}')
print(f'{"="*70}\n')

for ds in DATASETS:
    print(f'\n=== {ds} ===')
    for model_name in MODELS:
        print(f'  ▶  {model_name:<22} ', end='', flush=True)
        try:
            res = compare_shap_for_model(ds, model_name)
            results.append(res)
            print(f'ρ_global={res["spearman_global"]:.3f}, top5_overlap={res["top5_overlap_global"]}/5, {res["time_seconds"]:.1f}s')
        except Exception as e:
            print(f'❌ FAILED: {type(e).__name__}: {e}')
            errors.append({'dataset': ds, 'model': model_name, 'error': str(e), 'tb': traceback.format_exc()})

print(f'\nTotal time: {(time.time()-t_overall)/60:.1f} min')
print(f'Successful: {len(results)}, Failed: {len(errors)}')


SHAP validation experiment — 15:20:01


=== nsl_kdd_v2 ===
  ▶  rf_5class_cw           (reusing existing) ρ_global=0.880, top5_overlap=4/5, 0.0s
  ▶  xgb_5class_cw          ❌ FAILED: OSError: [Errno 107] Transport endpoint is not connected
  ▶  dnn_5class_cw          ❌ FAILED: DeferredCudaCallError: CUDA call failed lazily at initialization with error: [Errno 107] Transport endpoint is not connected

CUDA call was originally invoked at:

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py

## 6. Summary table and save

In [ ]:
if results:
    rows = []
    for r in results:
        rows.append({
            'dataset': r['dataset'],
            'model': r['model'],
            'model_type': r['model_type'],
            'spearman_global': r['spearman_global'],
            'top5_overlap_global': r['top5_overlap_global'],
            'spearman_class_0_Normal': r['per_class'][0]['spearman'],
            'spearman_class_1_DoS': r['per_class'][1]['spearman'],
            'spearman_class_2_Probe': r['per_class'][2]['spearman'],
            'spearman_class_3_R2L': r['per_class'][3]['spearman'],
            'spearman_class_4_U2R': r['per_class'][4]['spearman'],
            'top5_class_0_Normal': r['per_class'][0]['top5_overlap'],
            'top5_class_1_DoS': r['per_class'][1]['top5_overlap'],
            'top5_class_2_Probe': r['per_class'][2]['top5_overlap'],
            'top5_class_3_R2L': r['per_class'][3]['top5_overlap'],
            'top5_class_4_U2R': r['per_class'][4]['top5_overlap'],
            'time_seconds': r['time_seconds'],
        })
    df = pd.DataFrame(rows)

    out_csv = Path(REPO) / 'results' / 'tables' / 'shap_raw_vs_calibrated.csv'
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f'✓ Saved: {out_csv.relative_to(REPO)}')

    # Summary JSON
    all_spearman_global = [r['spearman_global'] for r in results if r['spearman_global'] is not None]
    all_top5_overlap = [r['top5_overlap_global'] for r in results]

    all_spearman_per_class = []
    for r in results:
        for c in range(5):
            v = r['per_class'][c]['spearman']
            if v is not None:
                all_spearman_per_class.append(v)

    summary = {
        'timestamp': datetime.now().isoformat(),
        'n_models': len(results),
        'n_eval_samples': N_EVAL,
        'n_background': N_BACKGROUND,
        'global_spearman_stats': {
            'mean': float(np.mean(all_spearman_global)),
            'min': float(np.min(all_spearman_global)),
            'max': float(np.max(all_spearman_global)),
            'median': float(np.median(all_spearman_global)),
        },
        'global_top5_overlap_stats': {
            'mean': float(np.mean(all_top5_overlap)),
            'min': int(np.min(all_top5_overlap)),
            'max': int(np.max(all_top5_overlap)),
            'count_5of5': int(sum(1 for v in all_top5_overlap if v == 5)),
            'count_4_or_better': int(sum(1 for v in all_top5_overlap if v >= 4)),
        },
        'per_class_spearman_stats': {
            'mean': float(np.mean(all_spearman_per_class)),
            'min': float(np.min(all_spearman_per_class)),
            'median': float(np.median(all_spearman_per_class)),
            'count_above_0.9': int(sum(1 for v in all_spearman_per_class if v >= 0.9)),
            'count_above_0.8': int(sum(1 for v in all_spearman_per_class if v >= 0.8)),
            'total': len(all_spearman_per_class),
        },
    }

    out_json = Path(REPO) / 'results' / 'tables' / 'shap_validation_summary.json'
    with open(out_json, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'✓ Saved: {out_json.relative_to(REPO)}')

    print('\n=== Summary ===')
    print(json.dumps(summary, indent=2))

    print('\n=== Full table ===')
    print(df.to_string(index=False))

## 7. Interpret the result

In [ ]:
if results:
    mean_rho = summary['global_spearman_stats']['mean']
    min_rho = summary['global_spearman_stats']['min']
    mean_top5 = summary['global_top5_overlap_stats']['mean']

    print('=' * 60)
    print('INTERPRETATION')
    print('=' * 60)

    print(f'\nGlobal Spearman (raw vs calibrated feature importance):')
    print(f'  Mean: {mean_rho:.3f}')
    print(f'  Min:  {min_rho:.3f}')

    print(f'\nTop-5 global feature overlap:')
    print(f'  Mean: {mean_top5:.2f}/5')

    print('\nInterpretation:')
    if mean_rho >= 0.9 and mean_top5 >= 4:
        print('✓ STRONG: Feature importance rankings are robust to calibration.')
        print('  Paper claim: "SHAP computed on raw model probabilities suffices for')
        print('  interpretation; calibration is a per-class monotonic transformation')
        print('  that preserves feature attribution rankings."')
    elif mean_rho >= 0.7:
        print('⚠ MODERATE: Rankings broadly agree but with notable divergence.')
        print('  Paper claim: Report both raw and calibrated SHAP rankings.')
        print('  Discuss specific cases where they diverge.')
    else:
        print('❌ WEAK: Calibration significantly alters feature attribution rankings.')
        print('  This is a DIFFERENT paper finding — calibration changes what features matter.')
        print('  All downstream interpretability claims need to be re-examined.')

## 8. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/04b_calibration_shap_validation.ipynb
!git add results/tables/shap_raw_vs_calibrated.csv
!git add results/tables/shap_validation_summary.json
!git status --short
!git commit -m 'Notebook 04b: SHAP raw-vs-calibrated validation experiment (9 models, 100 samples each)'
!git push origin main